# Central Differential Privacy Neural Network Training

Dataset: **Bank Marketing**

This notebook corresponds to **Section 6 "Private-model workflow
(Central Differential Privacy)"** of the paper.

It loads the preprocessed dataset (`bank-processed.csv`, produced by
`03_data_preprocessing.ipynb`), trains a neural network under Differentially
Private Stochastic Gradient Descent (DP-SGD) using TensorFlow Privacy, and
runs the grid search over sample size, batch size, noise multiplier, learning
rate, and clipping norm that produces Figures 5–16 of the paper.

> **Environment**: run this notebook in the `.cdp` virtual environment
> (TensorFlow 2.3.0 + tensorflow-privacy 0.5.1; see `requirements/cdp.txt`).
> The `.ldp` environment uses Keras 3 and will not work here.

In [2]:
import os
import random
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)
from imblearn.combine import SMOTEENN
from boruta import BorutaPy
import xgboost as xgb

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasSGDOptimizer
from tensorflow_privacy.privacy.analysis import compute_dp_sgd_privacy

import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

# Paths
ROOT = Path.cwd().resolve()
DATA_PROCESSED = ROOT.parent / "data" / "processed"
FIG_DIR = ROOT.parent / "figures"
RES_DIR = ROOT.parent / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Network architecture (fixed across the grid search)
HIDDEN_UNITS = 64
HIDDEN_LAYERS = 2
DROPOUT_RATE = 0.2
EPOCHS = 50

# Number of independent grid-search repetitions
N_REPEATS = 10


In [2]:
# Load preprocessed data and create train/test split
data = pd.read_csv(DATA_PROCESSED / "bank-processed.csv")
X = data.drop(columns=["y"])
y = data["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y,
)


In [3]:
# Class-balanced resampling on the training set
X_resample, y_resample = SMOTEENN(random_state=SEED).fit_resample(X_train, y_train)
X_resample = pd.DataFrame(X_resample, columns=X.columns)
y_resample = pd.Series(y_resample)


In [4]:
# Boruta feature selection on the resampled training set
xgbc = xgb.XGBClassifier(eval_metric="logloss", random_state=SEED)
feat_selector = BorutaPy(xgbc, n_estimators="auto", verbose=0, random_state=SEED)
feat_selector.fit(X_resample.values, y_resample.values.ravel())

if not feat_selector.support_.any():
    raise ValueError("Boruta did not select any features. Check preprocessing or model setup.")

X_filtered = X.columns[feat_selector.support_].tolist()

X_train_filtered = X_resample[X_filtered].values
X_test_filtered  = X_test[X_filtered].values
y_train_filtered = y_resample.values
y_test_filtered  = y_test.values

INPUT_SIZE = len(X_filtered)


c:\Users\danie\OneDrive\Documentos\1 UNIANDES\PAPER DP\differential-privacy-data-analysis\.venv-cdp\lib\site-packages\xgboost\sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


## Helper functions

In [5]:
# Compute the DP-SGD privacy budget for the given training configuration
def compute_privacy_budget(n, batch_size, noise_multiplier, epochs, delta=1e-5):
    try:
        return compute_dp_sgd_privacy.compute_dp_sgd_privacy(
            n=n, batch_size=batch_size, noise_multiplier=noise_multiplier,
            epochs=epochs, delta=delta,
        )[0]
    except Exception:
        return float("inf")


In [6]:
# Feedforward network with an optional DP-SGD optimizer (paper §6.1)
def create_model(input_size, hidden_units, hidden_layers, dropout_rate,
                 learning_rate, num_microbatches, l2_norm_clip,
                 noise_multiplier, use_dp):
    model = Sequential()
    model.add(Dense(hidden_units, activation="relu", input_shape=(input_size,)))
    for _ in range(hidden_layers - 1):
        model.add(Dense(hidden_units, activation="relu"))
        model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation="sigmoid"))

    if use_dp:
        optimizer = DPKerasSGDOptimizer(
            l2_norm_clip=l2_norm_clip,
            noise_multiplier=noise_multiplier,
            num_microbatches=num_microbatches,
            learning_rate=learning_rate,
        )
    else:
        optimizer = Adam(learning_rate=learning_rate)

    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return model


In [7]:
# Train one model on the given fold and return its test-set predictions
def train_model(X_train, y_train, X_test, y_test, batch_size, epochs,
                learning_rate, use_dp, noise_multiplier, l2_norm_clip,
                use_early_stopping):
    model = create_model(
        input_size=INPUT_SIZE,
        hidden_units=HIDDEN_UNITS,
        hidden_layers=HIDDEN_LAYERS,
        dropout_rate=DROPOUT_RATE,
        learning_rate=learning_rate,
        num_microbatches=batch_size,
        l2_norm_clip=l2_norm_clip,
        noise_multiplier=noise_multiplier,
        use_dp=use_dp,
    )

    callbacks = []
    if use_early_stopping:
        callbacks.append(EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True))

    model.fit(
        X_train, y_train,
        batch_size=batch_size,
        epochs=epochs,
        validation_split=0.2,
        callbacks=callbacks,
        verbose=0,
    )

    y_prob = model.predict(X_test, batch_size=batch_size).flatten()
    y_pred = (y_prob > 0.5).astype(int)
    return y_prob, y_pred


In [8]:
# Compute the metric panel used in the paper from test-set predictions
def evaluate_model(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    neg_total = cm[0].sum()
    pos_total = cm[1].sum()
    return {
        "ROC AUC":       roc_auc_score(y_true, y_prob),
        "Accuracy":      accuracy_score(y_true, y_pred),
        "Precision":     precision_score(y_true, y_pred),
        "Recall":        recall_score(y_true, y_pred),
        "F1 Score":      f1_score(y_true, y_pred),
        "Type I Error":  cm[0][1] / neg_total if neg_total > 0 else 0,
        "Type II Error": cm[1][0] / pos_total if pos_total > 0 else 0,
    }


In [9]:
# One full experiment: train, evaluate, and tag with its configuration
def run_experiment(X_train_data, y_train_data, batch_size, learning_rate,
                   noise_multiplier, l2_norm_clip,
                   use_dp=True, use_early_stopping=True):
    eps = compute_privacy_budget(
        n=len(X_train_data),
        batch_size=batch_size,
        noise_multiplier=noise_multiplier,
        epochs=EPOCHS,
    )
    y_prob, y_pred = train_model(
        X_train_data, y_train_data, X_test_filtered, y_test_filtered,
        batch_size=batch_size, epochs=EPOCHS, learning_rate=learning_rate,
        use_dp=use_dp, noise_multiplier=noise_multiplier, l2_norm_clip=l2_norm_clip,
        use_early_stopping=use_early_stopping,
    )
    results = evaluate_model(y_test_filtered, y_pred, y_prob)
    results.update({
        "Epsilon":          eps,
        "Batch Size":       batch_size,
        "Noise Multiplier": noise_multiplier,
        "Learning Rate":    learning_rate,
        "Clipping Norm":    l2_norm_clip,
        "Sample Ratio":     len(X_train_data) / len(X_train_filtered),
        "DP Enabled":       use_dp,
    })
    return results


In [10]:
# Full grid search: non-DP baseline and DP-SGD sweep over five hyperparameters,
# replicated across four sub-sampling ratios.
def grid_search_experiments(use_early_stopping=True):
    batch_sizes       = [16, 32, 64, 128]
    noise_multipliers = [0.8, 1.1, 1.5, 2.0]
    learning_rates    = [0.001, 0.003, 0.005]
    clip_norms        = [0.5, 1.0, 2.0]
    sample_ratios     = [1.0, 0.5, 0.1, 0.05]

    results = []
    for ratio in sample_ratios:
        n_samples = int(len(X_train_filtered) * ratio)
        idx = np.random.choice(len(X_train_filtered), n_samples, replace=False)
        X_sample = X_train_filtered[idx]
        y_sample = y_train_filtered[idx]

        # Non-DP baseline: varies batch size and learning rate only
        for bs, lr in product(batch_sizes, learning_rates):
            results.append(run_experiment(
                X_sample, y_sample,
                batch_size=bs, learning_rate=lr,
                noise_multiplier=0.0, l2_norm_clip=0.0,
                use_dp=False, use_early_stopping=use_early_stopping,
            ))

        # DP-SGD: varies batch size, noise multiplier, learning rate, clipping norm
        for bs, noise, lr, clip in product(
            batch_sizes, noise_multipliers, learning_rates, clip_norms,
        ):
            results.append(run_experiment(
                X_sample, y_sample,
                batch_size=bs, learning_rate=lr,
                noise_multiplier=noise, l2_norm_clip=clip,
                use_dp=True, use_early_stopping=use_early_stopping,
            ))

    return pd.DataFrame(results)


## Run grid search

In [11]:
# Repeat the entire grid search N_REPEATS times to absorb training stochasticity
all_runs = []
for run_id in range(N_REPEATS):
    print(f"--- Grid search run {run_id + 1}/{N_REPEATS} ---")
    df_run = grid_search_experiments(use_early_stopping=True)
    df_run["Run"] = run_id + 1
    all_runs.append(df_run)


--- Grid search run 1/10 ---
DP-SGD with sampling rate = 0.0299% and noise_multiplier = 0.8 iterated over 167200 steps satisfies differential privacy with eps = 1.7 and delta = 1e-05.
The optimal RDP order is 9.0.
DP-SGD with sampling rate = 0.0299% and noise_multiplier = 0.8 iterated over 167200 steps satisfies differential privacy with eps = 1.7 and delta = 1e-05.
The optimal RDP order is 9.0.
DP-SGD with sampling rate = 0.0299% and noise_multiplier = 0.8 iterated over 167200 steps satisfies differential privacy with eps = 1.7 and delta = 1e-05.
The optimal RDP order is 9.0.
DP-SGD with sampling rate = 0.0299% and noise_multiplier = 0.8 iterated over 167200 steps satisfies differential privacy with eps = 1.7 and delta = 1e-05.
The optimal RDP order is 9.0.
DP-SGD with sampling rate = 0.0299% and noise_multiplier = 0.8 iterated over 167200 steps satisfies differential privacy with eps = 1.7 and delta = 1e-05.
The optimal RDP order is 9.0.
DP-SGD with sampling rate = 0.0299% and noise_

In [12]:
# Combine runs and aggregate to mean/min/max per configuration
df_all = pd.concat(all_runs, ignore_index=True)
df_all.round(3).to_csv(RES_DIR / "cdp_all_runs.csv", index=False)

metrics = [
    "ROC AUC", "Accuracy", "Precision", "Recall",
    "F1 Score", "Type I Error", "Type II Error", "Epsilon",
]
group_cols = [
    "Batch Size", "Noise Multiplier", "Learning Rate",
    "Clipping Norm", "Sample Ratio", "DP Enabled",
]

agg_results = (
    df_all.groupby(group_cols)[metrics]
          .agg(["mean", "min", "max"])
          .reset_index()
)
agg_results.columns = [" ".join(col).strip() for col in agg_results.columns.values]
agg_results.round(3).to_csv(RES_DIR / "cdp_aggregated_results.csv", index=False)


## Plot results

In [3]:
# Plotting setup: style, output subfolders, reload aggregated results
plt.style.use("seaborn")

for sub in ("cdp1", "cdp2", "cdp3", "cdp4", "cdp5"):
    (FIG_DIR / sub).mkdir(parents=True, exist_ok=True)

df_results = pd.read_csv(RES_DIR / "cdp_aggregated_results.csv")
non_dp_data = df_results[df_results["DP Enabled"] == False]
dp_data     = df_results[df_results["DP Enabled"] == True]

# Grid axes used by the plots below
sample_ratios     = [1.0, 0.5, 0.1, 0.05]
batch_sizes       = [16, 32, 64, 128]
learning_rates    = [0.001, 0.003, 0.005]
noise_multipliers = [0.8, 1.1, 1.5, 2.0]
clip_norms        = [0.5, 1.0, 2.0]

VMIN, VMAX = 0.58, 0.9

In [4]:
# Plot 1: non-DP ROC AUC heatmaps per sample ratio (paper Figure 5)
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.ravel()
fig.suptitle("ROC AUC for non-DP experiments (bank marketing)",
             fontsize=18, fontweight="bold")

for i, ratio in enumerate(sample_ratios):
    subset = non_dp_data[non_dp_data["Sample Ratio"] == ratio]
    pivot = subset.pivot(index="Batch Size", columns="Learning Rate", values="ROC AUC mean")
    pivot = pivot.reindex(index=batch_sizes, columns=learning_rates, fill_value=np.nan)

    sns.heatmap(
        pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[i],
        cbar_kws={"label": "ROC AUC"}, annot_kws={"size": 13},
        vmin=VMIN, vmax=VMAX,
    )
    axes[i].set_title(f"Sample Ratio: {ratio}", fontsize=16)
    axes[i].set_xlabel("Learning Rate", fontsize=14)
    axes[i].set_ylabel("Batch Size", fontsize=14)
    axes[i].tick_params(axis="both", labelsize=12)
    cbar = axes[i].collections[0].colorbar
    cbar.ax.tick_params(labelsize=12)
    cbar.set_label("ROC AUC", fontsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(FIG_DIR / "cdp1" / "non_dp_roc_auc_heatmaps.png")
plt.close()


In [5]:
# Plot 2: DP epsilon heatmaps per sample ratio (paper Figure 7)
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.ravel()
fig.suptitle("Epsilon for DP experiments by sample ratio (bank marketing)",
             fontsize=18, fontweight="bold")

for i, ratio in enumerate(sample_ratios):
    subset = dp_data[dp_data["Sample Ratio"] == ratio]
    pivot = subset.pivot_table(
        index="Batch Size", columns="Noise Multiplier",
        values="Epsilon mean", aggfunc="mean",
    )
    pivot = pivot.reindex(index=batch_sizes, columns=noise_multipliers, fill_value=np.nan)

    sns.heatmap(
        pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[i],
        cbar_kws={"label": "Epsilon"}, annot_kws={"size": 13},
    )
    axes[i].set_title(f"Sample Ratio: {ratio}", fontsize=16)
    axes[i].set_xlabel("Noise Multiplier", fontsize=14)
    axes[i].set_ylabel("Batch Size", fontsize=14)
    axes[i].tick_params(axis="both", labelsize=12)
    cbar = axes[i].collections[0].colorbar
    cbar.ax.tick_params(labelsize=12)
    cbar.set_label("Epsilon", fontsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(FIG_DIR / "cdp2" / "dp_epsilon_heatmaps.png")
plt.close()


In [6]:
# Plot 3: DP ROC AUC heatmaps at batch_size=16 across four noise multipliers
# (paper Figure 9). For each fixed (batch, noise), the panel sweeps over
# sample ratios; each cell varies clipping norm vs. learning rate.
configurations = [{"batch_size": 16, "noise_multiplier": nm}
                    for nm in noise_multipliers]

for config in configurations:
    bs = config["batch_size"]
    nm = config["noise_multiplier"]
    config_data = dp_data[(dp_data["Batch Size"] == bs) & (dp_data["Noise Multiplier"] == nm)]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()
    fig.suptitle(
        f"ROC AUC for DP (bank marketing), Batch Size={bs}, Noise Multiplier={nm}",
        fontsize=18, fontweight="bold",
    )

    for i, ratio in enumerate(sample_ratios):
        subset = config_data[config_data["Sample Ratio"] == ratio]
        epsilon = subset["Epsilon mean"].mean() if not subset.empty else np.nan
        epsilon_str = f"{epsilon:.3f}" if not np.isnan(epsilon) else "N/A"

        pivot = subset.pivot(index="Clipping Norm", columns="Learning Rate", values="ROC AUC mean")
        pivot = pivot.reindex(index=clip_norms, columns=learning_rates, fill_value=np.nan)

        sns.heatmap(
            pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[i],
            cbar_kws={"label": "ROC AUC"}, vmin=VMIN, vmax=VMAX, annot_kws={"size": 13},
        )
        axes[i].set_title(f"Sample Ratio: {ratio}, Epsilon: {epsilon_str}", fontsize=16)
        axes[i].set_xlabel("Learning Rate", fontsize=14)
        axes[i].set_ylabel("Clipping Norm", fontsize=14)
        axes[i].tick_params(axis="both", labelsize=12)
        cbar = axes[i].collections[0].colorbar
        cbar.ax.tick_params(labelsize=12)
        cbar.set_label("ROC AUC", fontsize=14)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(FIG_DIR / "cdp3" / f"dp_roc_auc_bs_{bs}_nm_{nm}_sr_{ratio}.png")
    plt.close()


In [7]:
# Plot 4: For each sample ratio, ROC AUC heatmaps at batch_size=16 across the
# four noise multipliers (paper Figures 11, 13)
noise_configurations = [{"batch_size": 16, "noise_multiplier": nm}
                        for nm in noise_multipliers]

for ratio in sample_ratios:
    ratio_data = dp_data[dp_data["Sample Ratio"] == ratio]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()
    fig.suptitle(
        f"ROC AUC for DP (bank marketing), Sample Ratio={ratio}",
        fontsize=18, fontweight="bold",
    )

    for i, cfg in enumerate(noise_configurations):
        bs, nm = cfg["batch_size"], cfg["noise_multiplier"]
        subset = ratio_data[(ratio_data["Batch Size"] == bs) & (ratio_data["Noise Multiplier"] == nm)]
        epsilon = subset["Epsilon mean"].mean() if not subset.empty else np.nan
        epsilon_str = f"{epsilon:.3f}" if not np.isnan(epsilon) else "N/A"

        pivot = subset.pivot(index="Clipping Norm", columns="Learning Rate", values="ROC AUC mean")
        pivot = pivot.reindex(index=clip_norms, columns=learning_rates, fill_value=np.nan)

        sns.heatmap(
            pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[i],
            cbar_kws={"label": "ROC AUC"}, vmin=VMIN, vmax=VMAX, annot_kws={"size": 13},
        )
        axes[i].set_title(f"Batch Size: {bs}, Noise: {nm}\nEpsilon: {epsilon_str}", fontsize=16)
        axes[i].set_xlabel("Learning Rate", fontsize=14)
        axes[i].set_ylabel("Clipping Norm", fontsize=14)
        axes[i].tick_params(axis="both", labelsize=12)
        cbar = axes[i].collections[0].colorbar
        cbar.ax.tick_params(labelsize=12)
        cbar.set_label("ROC AUC", fontsize=14)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(FIG_DIR / "cdp4" / f"dp_roc_auc_bs_{bs}_nm_{nm}_sr_{ratio}.png")
    plt.close()


In [8]:
# Plot 5: For each sample ratio, ROC AUC heatmaps at noise_multiplier=1.5 across
# the four batch sizes (paper Figure 15)
batch_configurations = [{"batch_size": bs, "noise_multiplier": 1.5}
                        for bs in batch_sizes]

for ratio in sample_ratios:
    ratio_data = dp_data[dp_data["Sample Ratio"] == ratio]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()
    fig.suptitle(
        f"ROC AUC for DP (bank marketing), Sample Ratio={ratio}",
        fontsize=18, fontweight="bold",
    )

    for i, cfg in enumerate(batch_configurations):
        bs, nm = cfg["batch_size"], cfg["noise_multiplier"]
        subset = ratio_data[(ratio_data["Batch Size"] == bs) & (ratio_data["Noise Multiplier"] == nm)]
        epsilon = subset["Epsilon mean"].mean() if not subset.empty else np.nan
        epsilon_str = f"{epsilon:.3f}" if not np.isnan(epsilon) else "N/A"

        pivot = subset.pivot(index="Clipping Norm", columns="Learning Rate", values="ROC AUC mean")
        pivot = pivot.reindex(index=clip_norms, columns=learning_rates, fill_value=np.nan)

        sns.heatmap(
            pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[i],
            cbar_kws={"label": "ROC AUC"}, vmin=VMIN, vmax=VMAX, annot_kws={"size": 13},
        )
        axes[i].set_title(f"Batch Size: {bs}, Noise: {nm}\nEpsilon: {epsilon_str}", fontsize=16)
        axes[i].set_xlabel("Learning Rate", fontsize=14)
        axes[i].set_ylabel("Clipping Norm", fontsize=14)
        axes[i].tick_params(axis="both", labelsize=12)
        cbar = axes[i].collections[0].colorbar
        cbar.ax.tick_params(labelsize=12)
        cbar.set_label("ROC AUC", fontsize=14)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(FIG_DIR / "cdp5" / f"dp_roc_auc_bs_{bs}_nm_{nm}_sr_{ratio}.png")
    plt.close()


## ANOVA

In [9]:
# Non-DP ANOVA: 2-way partial interactions over batch size, sample ratio, learning rate
# (paper Table 2)
anova_non_dp_data = non_dp_data[
    ["ROC AUC mean", "Batch Size", "Learning Rate", "Sample Ratio"]
].astype({
    "Batch Size":    "category",
    "Learning Rate": "category",
    "Sample Ratio":  "category",
})

formula_non_dp = (
    'Q("ROC AUC mean") ~ '
    'C(Q("Batch Size")) * C(Q("Sample Ratio")) + '
    'C(Q("Batch Size")) * C(Q("Learning Rate")) + '
    'C(Q("Sample Ratio")) * C(Q("Learning Rate"))'
)

model_non_dp = smf.ols(formula=formula_non_dp, data=anova_non_dp_data).fit()
anova_lm(model_non_dp, typ=2)


,sum_sq,df,F,PR(>F)
"C(Q(""Batch Size""))",3.156250e-05,3.0,23.795812,1.735987e-06
"C(Q(""Sample Ratio""))",3.862729e-03,3.0,2912.214660,2.311706e-24
"C(Q(""Learning Rate""))",2.916667e-07,2.0,0.329843,7.232919e-01
"C(Q(""Batch Size"")):C(Q(""Sample Ratio""))",3.435417e-05,9.0,8.633508,6.249050e-05
"C(Q(""Batch Size"")):C(Q(""Learning Rate""))",1.187500e-05,6.0,4.476440,6.073218e-03
"C(Q(""Sample Ratio"")):C(Q(""Learning Rate""))",1.208333e-06,6.0,0.455497,8.317672e-01
Residual,7.958333e-06,18.0,NaN,NaN


In [10]:
# DP ANOVA: main effects over all five DP-SGD hyperparameters (paper Table 3)
anova_dp_data = dp_data[
    ["ROC AUC mean", "Batch Size", "Noise Multiplier", "Clipping Norm",
     "Learning Rate", "Sample Ratio"]
].astype({
    "Batch Size":       "category",
    "Noise Multiplier": "category",
    "Clipping Norm":    "category",
    "Learning Rate":    "category",
    "Sample Ratio":     "category",
})

formula_dp = (
    'Q("ROC AUC mean") ~ '
    'C(Q("Batch Size")) + C(Q("Noise Multiplier")) + '
    'C(Q("Clipping Norm")) + C(Q("Learning Rate")) + C(Q("Sample Ratio"))'
)

model_dp = smf.ols(formula=formula_dp, data=anova_dp_data).fit()
anova_lm(model_dp, typ=2)


,sum_sq,df,F,PR(>F)
"C(Q(""Batch Size""))",0.577754,3.0,732.489739,1.076606e-193
"C(Q(""Noise Multiplier""))",0.000007,3.0,0.008353,9.989515e-01
"C(Q(""Clipping Norm""))",0.000039,2.0,0.074095,9.285925e-01
"C(Q(""Learning Rate""))",0.436163,2.0,829.465890,1.993005e-168
"C(Q(""Sample Ratio""))",2.052662,3.0,2602.413146,0.000000e+00
Residual,0.147760,562.0,NaN,NaN
